In [2]:
# Importing
import pandas as pd
import numpy as np
from utils.fake_na_detection_and_cleaning import detect_fake_nulls,replace_fake_nulls
from utils.sql_connector import connect_sql

In [3]:
# loading of 2021 datasets
query='select * from Bronze.Survey_2021'
database='Stack_Overflow_Survey'
conn = connect_sql(database)
print(conn)
Survey_2021_df = pd.read_sql(query,conn)

Succesfully Connected


C:\Users\Ayush\AppData\Local\Temp\ipykernel_27556\2218414459.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  Survey_2021_df = pd.read_sql(query,conn)


In [4]:
detect_fake_nulls(Survey_2021_df)
replace_fake_nulls(Survey_2021_df)

categorical_cols = [
    "MainBranch", "Employment", "EdLevel", "Age", "Age1stCode",
    "LearnCode", "OpSys", "OrgSize", "Country", "US_State", "UK_Country",
    "Gender", "Trans", "Sexuality", "Ethnicity", "Accessibility",
    "MentalHealth", "SOVisitFreq", "SOAccount", "SOPartFreq", "SOComm",
    "NEWSOSites", "NEWOtherComms", "NEWStuck", "CompFreq",
    "SurveyLength", "SurveyEase", "DevType"
]
# for col in categorical_cols:
#     Survey_2021_df[col]=Survey_2021_df[col].astype("category")


{'Accessibility': {'NA': np.int64(5836)},
 'Age': {'NA': np.int64(1032)},
 'Age1stCode': {'NA': np.int64(196)},
 'CompFreq': {'NA': np.int64(31289)},
 'CompTotal': {'NA': np.int64(36256)},
 'ConvertedCompYearly': {'NA': np.int64(36595)},
 'Currency': {'NA': np.int64(22359)},
 'DatabaseHaveWorkedWith': {'NA': np.int64(13893)},
 'DatabaseWantToWorkWith': {'NA': np.int64(25140)},
 'DevType': {'NA': np.int64(16955)},
 'EdLevel': {'NA': np.int64(313)},
 'Employment': {'NA': np.int64(116)},
 'Ethnicity': {'NA': np.int64(3975)},
 'Gender': {'NA': np.int64(1153)},
 'LanguageHaveWorkedWith': {'NA': np.int64(1082)},
 'LanguageWantToWorkWith': {'NA': np.int64(6618)},
 'LearnCode': {'NA': np.int64(476)},
 'MentalHealth': {'NA': np.int64(6519)},
 'MiscTechHaveWorkedWith': {'NA': np.int64(36384)},
 'MiscTechWantToWorkWith': {'NA': np.int64(45418)},
 'NEWCollabToolsHaveWorkedWith': {'NA': np.int64(2205)},
 'NEWCollabToolsWantToWorkWith': {'NA': np.int64(10417)},
 'NEWOtherComms': {'NA': np.int64(611)

In [ ]:
# Mormalization of categorical columns

employment_map = {
    'Employed full-time': 'Employed',
    'Employed part-time': 'Employed',
    'Independent contractor, freelancer, or self-employed': 'Freelance',
    'Student, full-time': 'Student',
    'Student, part-time': 'Student',
    'Not employed, but looking for work': 'Unemployed',
    'Not employed, and not looking for work': 'Unemployed',
    'Retired': 'Retired',
    'I prefer not to say': 'I prefer not to say',
    'Nan' : 'Not Available',
}
ed_level_map = {
    'BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)': 'Undergraduate',
    'MasterΓÇÖs degree (M.A., M.S., M.Eng., MBA, etc.)': 'Postgraduate',
    'Some college/university study without earning a degree': 'Undergraduate',
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 'High School',
    'Other doctoral degree (Ph.D., Ed.D., etc.)': 'Doctorate',
    'Primary/elementary school': 'Primary',
    'Associate degree (A.A., A.S., etc.)': 'Undergraduate',
    'Something else': 'Other',
    'Professional degree (JD, MD, etc.)': 'Postgraduate'
}
age_1st_code_map = {
    'Younger than 5 years': '<5',
    '5 - 10 years': '5-10',
    '11 - 17 years': '11-17',
    '18 - 24 years': '18-24',
    '25 - 34 years': '25-34',
    '35 - 44 years': '35-44',
    '45 - 54 years': '45-54',
    '55 - 64 years': '55-64',
    'Older than 64 years': '>64'
}
age_map = {
    'Under 18 years old': '<18',
    '18-24 years old': '18-24',
    '25-34 years old': '25-34',
    '35-44 years old': '35-44',
    '45-54 years old': '45-54',
    '55-64 years old': '55-64',
    '65 years or older': '>65',
    'Prefer not to say': 'Unknown'
}
un_normalized_cols_name = [
    'Employment', 'EdLevel', 'Age', 'Age1stCode', 'LearnCode', 'OpSys',
]
normalized_cols_name=[
    'Employment','Education_Level','Age','AgeCode','LearnCode','OperatingSystem'
]
columns_map=[
    employment_map, ed_level_map, age_map, age_1st_code_map
]

for un_normalized_col,normalized_col, col_map in zip(
    un_normalized_cols_name,
    normalized_cols_name, 
    columns_map
):
    # print(Survey_2021_df[un_normalized_col].unique())
    Survey_2021_df[un_normalized_col].value_counts(dropna=False)
    Survey_2021_df[normalized_col] = Survey_2021_df[un_normalized_col].map(col_map)
    print(Survey_2021_df[normalized_col].value_counts(dropna=False))


In [ ]:
print(Survey_2021_df['Age'].unique())
Survey_2021_df['Age'].value_counts(dropna=False)


Survey_2021_df['Age']=Survey_2021_df['Age'].map(age_map).fillna("Not Available")
Survey_2021_df['Age'].value_counts(dropna=False)

['25-34 years old' '18-24 years old' '35-44 years old' 'Prefer not to say'
 '45-54 years old' 'Under 18 years old' '65 years or older'
 '55-64 years old' nan]


Age
25-34            32568
18-24            20993
35-44            15183
45-54             5472
<18               5376
55-64             1819
Not Available     1032
Unknown            575
>65                421
Name: count, dtype: int64